In [1]:
# ============================================================
# AUDIT BEFORE FIXING RESPICAST COMPARISON
# Checks:
# 1) duplicates in finaltest_all_predictions_long.csv
# 2) coverage > 1 cause
# 3) horizon convention in RespiCast model-output files
# 4) date overlap between TFG and RespiCast
# ============================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import re

try:
    from IPython.display import display
except Exception:
    display = print

# ------------------------------------------------------------
# Locate folders
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.config import get_project_paths
    paths = get_project_paths(PROJECT_ROOT)
    RESULTS_DIR = paths.results
except Exception:
    RESULTS_DIR = PROJECT_ROOT / "results"

FINAL_TEST_DIR = RESULTS_DIR / "final_test_2024_2025"

RESPICAST_REPO = PROJECT_ROOT / "RespiCast-SyndromicIndicators-main"
MODEL_OUTPUT_DIR = RESPICAST_REPO / "model-output"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_TEST_DIR:", FINAL_TEST_DIR)
print("MODEL_OUTPUT_DIR:", MODEL_OUTPUT_DIR)

pred_path = FINAL_TEST_DIR / "finaltest_all_predictions_long.csv"

if not pred_path.exists():
    raise FileNotFoundError(pred_path)

if not MODEL_OUTPUT_DIR.exists():
    raise FileNotFoundError(MODEL_OUTPUT_DIR)

# ------------------------------------------------------------
# 1) Load TFG final predictions
# ------------------------------------------------------------
df = pd.read_csv(pred_path)

df.columns = [c.strip() for c in df.columns]

required = ["origin", "target", "disease", "location", "h", "y", "pred", "model"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["origin"] = pd.to_datetime(df["origin"])
df["target"] = pd.to_datetime(df["target"])
df["disease"] = df["disease"].astype(str)
df["location"] = df["location"].astype(str)
df["model"] = df["model"].astype(str)
df["h"] = pd.to_numeric(df["h"], errors="coerce").astype(int)
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df["pred"] = pd.to_numeric(df["pred"], errors="coerce")

print("\n============================================================")
print("TFG final predictions loaded")
print("============================================================")
print("Shape:", df.shape)
display(df.head())

# ------------------------------------------------------------
# 2) Check duplicate prediction keys
# ------------------------------------------------------------
key_cols = ["origin", "target", "disease", "location", "h", "model"]

dup_stats = (
    df.groupby(key_cols, as_index=False)
      .agg(
          n_rows=("pred", "size"),
          n_unique_pred=("pred", "nunique"),
          n_unique_y=("y", "nunique"),
          first_pred=("pred", "first"),
          last_pred=("pred", "last")
      )
)

dups = dup_stats[dup_stats["n_rows"] > 1].copy()

print("\n============================================================")
print("Duplicate prediction-key audit")
print("============================================================")
print("Number of duplicated prediction keys:", len(dups))
print("Duplicated raw rows:", int(dups["n_rows"].sum()) if len(dups) else 0)

if len(dups) > 0:
    print("\nDuplicates by model/disease/h:")
    dup_summary = (
        dups.groupby(["model", "disease", "h"], as_index=False)
            .agg(
                n_duplicate_keys=("n_rows", "size"),
                duplicated_rows=("n_rows", "sum"),
                max_rows_per_key=("n_rows", "max"),
                n_conflicting_pred_keys=("n_unique_pred", lambda x: int((x > 1).sum())),
                n_conflicting_y_keys=("n_unique_y", lambda x: int((x > 1).sum()))
            )
            .sort_values(["model", "disease", "h"])
    )
    display(dup_summary)

    print("\nSample duplicated keys:")
    display(dups.head(30))

    conflict_pred = dups[dups["n_unique_pred"] > 1].copy()
    conflict_y = dups[dups["n_unique_y"] > 1].copy()

    print("\nDuplicated keys with different predictions:", len(conflict_pred))
    print("Duplicated keys with different y:", len(conflict_y))

    if len(conflict_pred) > 0:
        print("\nSample conflicts in predictions:")
        display(conflict_pred.head(20))

    if len(conflict_y) > 0:
        print("\nSample conflicts in y:")
        display(conflict_y.head(20))
else:
    print("No duplicates found.")

# ------------------------------------------------------------
# 3) Coverage audit using unique keys, not raw rows
# ------------------------------------------------------------
truth_key_cols = ["origin", "target", "disease", "location", "h"]

truth_universe = (
    df[truth_key_cols + ["y"]]
    .dropna(subset=["y"])
    .drop_duplicates(truth_key_cols)
    .copy()
)

coverage_raw = (
    df.groupby(["model", "disease", "h"], as_index=False)
      .agg(raw_rows=("pred", "size"))
)

coverage_unique = (
    df.drop_duplicates(key_cols)
      .groupby(["model", "disease", "h"], as_index=False)
      .agg(unique_prediction_keys=("pred", "size"))
)

universe = (
    truth_universe.groupby(["disease", "h"], as_index=False)
                  .agg(universe_points=("y", "size"))
)

coverage = (
    coverage_raw
    .merge(coverage_unique, on=["model", "disease", "h"], how="left")
    .merge(universe, on=["disease", "h"], how="left")
)

coverage["raw_coverage_rate"] = coverage["raw_rows"] / coverage["universe_points"]
coverage["unique_coverage_rate"] = coverage["unique_prediction_keys"] / coverage["universe_points"]
coverage["extra_raw_rows_due_to_duplicates"] = coverage["raw_rows"] - coverage["unique_prediction_keys"]

print("\n============================================================")
print("Coverage audit: raw rows vs unique prediction keys")
print("============================================================")
display(
    coverage.sort_values(
        ["disease", "h", "raw_coverage_rate", "model"],
        ascending=[True, True, False, True]
    )
)

print("\nRows with raw coverage > 1:")
display(coverage[coverage["raw_coverage_rate"] > 1].sort_values(["disease", "h", "model"]))

print("\nRows with unique coverage > 1:")
display(coverage[coverage["unique_coverage_rate"] > 1].sort_values(["disease", "h", "model"]))

# ------------------------------------------------------------
# 4) Build a clean TFG table defensively, only if safe
# ------------------------------------------------------------
if len(dups) == 0:
    print("\nNo duplicated keys. Clean table equals original.")
else:
    n_conflicting_pred = int((dups["n_unique_pred"] > 1).sum())
    n_conflicting_y = int((dups["n_unique_y"] > 1).sum())

    if n_conflicting_pred == 0 and n_conflicting_y == 0:
        df_clean = (
            df.sort_values(key_cols)
              .drop_duplicates(key_cols, keep="first")
              .reset_index(drop=True)
        )

        clean_path = FINAL_TEST_DIR / "finaltest_all_predictions_long_DEDUP_SAFE.csv"
        df_clean.to_csv(clean_path, index=False)

        print("\nSafe deduplication possible: duplicated rows are identical.")
        print("Saved clean file to:", clean_path)
        print("Original shape:", df.shape)
        print("Clean shape:", df_clean.shape)
    else:
        print("\nWARNING: Some duplicated keys have conflicting pred or y values.")
        print("Do NOT deduplicate automatically yet. Inspect conflicts first.")

# ------------------------------------------------------------
# 5) Audit RespiCast horizon convention
# ------------------------------------------------------------
def find_col(cols, candidates):
    lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

def parse_date_from_filename(name):
    m = re.search(r"(\d{4}-\d{2}-\d{2})", name)
    if m:
        return pd.to_datetime(m.group(1))
    return pd.NaT

# Use a small but representative subset:
# reference baseline, hub ensemble, ECDC SARIMA if available.
priority_models = [
    "respicast-quantileBaseline",
    "respicast-hubEnsemble",
    "ECDC-SARIMA",
    "Chronos-Chronos2"
]

sample_files = []

for model_name in priority_models:
    model_dir = MODEL_OUTPUT_DIR / model_name
    if model_dir.exists():
        csvs = sorted(model_dir.glob("*.csv"))
        if len(csvs) > 0:
            sample_files.extend(csvs[:3])
            sample_files.extend(csvs[-3:])

# fallback
if len(sample_files) == 0:
    sample_files = sorted(MODEL_OUTPUT_DIR.rglob("*.csv"))[:20]

sample_files = list(dict.fromkeys(sample_files))

horizon_rows = []

for path in sample_files:
    try:
        tmp = pd.read_csv(path, low_memory=False)
    except Exception as e:
        print("Could not read:", path, e)
        continue

    tmp.columns = [c.strip() for c in tmp.columns]

    origin_col = find_col(tmp.columns, ["origin_date", "forecast_date", "reference_date"])
    target_col = find_col(tmp.columns, ["target_end_date", "target_date", "date"])
    horizon_col = find_col(tmp.columns, ["horizon"])
    disease_col = find_col(tmp.columns, ["target"])
    location_col = find_col(tmp.columns, ["location"])
    output_type_col = find_col(tmp.columns, ["output_type"])
    output_type_id_col = find_col(tmp.columns, ["output_type_id"])
    value_col = find_col(tmp.columns, ["value"])

    if origin_col is None or target_col is None:
        continue

    sub = tmp.copy()

    if output_type_col is not None:
        ot = sub[output_type_col].astype(str).str.lower()
        if output_type_id_col is not None:
            oti = pd.to_numeric(sub[output_type_id_col], errors="coerce")
            sub = sub[
                ot.eq("median") |
                ot.eq("mean") |
                ot.eq("point") |
                (ot.eq("quantile") & np.isclose(oti, 0.5, atol=1e-10))
            ].copy()
        else:
            sub = sub[ot.isin(["median", "mean", "point"])].copy()

    if len(sub) == 0:
        continue

    origin = pd.to_datetime(sub[origin_col], errors="coerce")
    target = pd.to_datetime(sub[target_col], errors="coerce")

    date_diff_weeks = ((target - origin).dt.days / 7.0)

    out = pd.DataFrame({
        "model": path.parent.name,
        "file": path.name,
        "file_date": parse_date_from_filename(path.name),
        "origin_raw": origin,
        "target_raw": target,
        "date_diff_weeks": date_diff_weeks,
    })

    if horizon_col is not None:
        out["horizon_column_raw"] = sub[horizon_col].astype(str).values
        out["horizon_column_num"] = pd.to_numeric(sub[horizon_col], errors="coerce").values
    else:
        out["horizon_column_raw"] = np.nan
        out["horizon_column_num"] = np.nan

    if disease_col is not None:
        out["target_name"] = sub[disease_col].astype(str).values
    else:
        out["target_name"] = ""

    if location_col is not None:
        out["location"] = sub[location_col].astype(str).values
    else:
        out["location"] = ""

    horizon_rows.append(out)

if len(horizon_rows) == 0:
    print("\nNo RespiCast horizon rows could be audited.")
else:
    horizon_audit = pd.concat(horizon_rows, ignore_index=True)

    print("\n============================================================")
    print("RespiCast horizon/date convention audit")
    print("============================================================")

    print("\nDistribution of date_diff_weeks:")
    display(
        horizon_audit.groupby(["model", "date_diff_weeks"], as_index=False)
                     .size()
                     .sort_values(["model", "date_diff_weeks"])
    )

    print("\nDistribution of horizon column:")
    display(
        horizon_audit.groupby(["model", "horizon_column_raw", "horizon_column_num"], dropna=False, as_index=False)
                     .size()
                     .sort_values(["model", "horizon_column_num", "horizon_column_raw"])
    )

    print("\nCross-tab: horizon column vs date difference:")
    display(
        horizon_audit.groupby(["model", "horizon_column_num", "date_diff_weeks"], dropna=False, as_index=False)
                     .size()
                     .sort_values(["model", "horizon_column_num", "date_diff_weeks"])
    )

    print("\nSample rows:")
    display(horizon_audit.head(40))

# ------------------------------------------------------------
# 6) Compare TFG h convention directly
# ------------------------------------------------------------
print("\n============================================================")
print("TFG horizon/date convention audit")
print("============================================================")

tfg_h_audit = df.copy()
tfg_h_audit["date_diff_weeks"] = ((tfg_h_audit["target"] - tfg_h_audit["origin"]).dt.days / 7.0)

display(
    tfg_h_audit.groupby(["h", "date_diff_weeks"], as_index=False)
               .size()
               .sort_values(["h", "date_diff_weeks"])
)

print("\nDone. Send me these displayed tables, especially:")
print("1) Duplicate prediction-key audit")
print("2) Coverage audit")
print("3) RespiCast horizon/date convention audit")

PROJECT_ROOT: C:\Users\aolas\UNI PYTHON\TFG
FINAL_TEST_DIR: C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025
MODEL_OUTPUT_DIR: C:\Users\aolas\UNI PYTHON\TFG\RespiCast-SyndromicIndicators-main\model-output

TFG final predictions loaded
Shape: (49892, 14)


,origin,target,disease,location,h,y,pred,model,fit_ok,converged,fit_error,selected_order,selected_seasonal_order,trend
0,2024-01-07,2024-01-14,ARI,BE,1,1106.1,1018.349334,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",1.0,1.0,NaN,"(1, 0, 0)","(1, 0, 0, 52)",c
1,2024-01-07,2024-01-21,ARI,BE,2,1237.4,1000.057044,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",1.0,1.0,NaN,"(1, 0, 0)","(1, 0, 0, 52)",c
2,2024-01-07,2024-01-28,ARI,BE,3,1515.6,1019.114361,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",1.0,1.0,NaN,"(1, 0, 0)","(1, 0, 0, 52)",c
3,2024-01-07,2024-02-04,ARI,BE,4,1699.8,1060.060397,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",1.0,1.0,NaN,"(1, 0, 0)","(1, 0, 0, 52)",c
4,2024-01-14,2024-01-21,ARI,BE,1,1237.4,1078.682112,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",1.0,1.0,NaN,"(1, 0, 0)","(1, 0, 0, 52)",c



Duplicate prediction-key audit
Number of duplicated prediction keys: 0
Duplicated raw rows: 0
No duplicates found.

Coverage audit: raw rows vs unique prediction keys


,model,disease,h,raw_rows,unique_prediction_keys,universe_points,raw_coverage_rate,unique_coverage_rate,extra_raw_rows_due_to_duplicates
0,DL_GlobalGRU_all_infections_all_countries,ARI,1,808,808,791,1.021492,1.021492,0
8,RandomForest(lags=4),ARI,1,808,808,791,1.021492,1.021492,0
16,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",ARI,1,808,808,791,1.021492,1.021492,0
24,autoARIMA,ARI,1,808,808,791,1.021492,1.021492,0
48,rf_global_all_infections_all_countries,ARI,1,808,808,791,1.021492,1.021492,0
32,ensemble_mean_5,ARI,1,791,791,791,1.000000,1.000000,0
40,ensemble_median_5,ARI,1,791,791,791,1.000000,1.000000,0
1,DL_GlobalGRU_all_infections_all_countries,ARI,2,800,800,783,1.021711,1.021711,0
9,RandomForest(lags=4),ARI,2,800,800,783,1.021711,1.021711,0
17,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",ARI,2,800,800,783,1.021711,1.021711,0



Rows with raw coverage > 1:


...


Sample rows:


,model,file,file_date,origin_raw,target_raw,date_diff_weeks,horizon_column_raw,horizon_column_num,target_name,location
0,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-10-20,-0.428571,1,1,ILI incidence,AT
1,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-10-20,-0.428571,1,1,ILI incidence,AT
2,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-10-27,0.571429,2,2,ILI incidence,AT
3,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-10-27,0.571429,2,2,ILI incidence,AT
4,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-11-03,1.571429,3,3,ILI incidence,AT
5,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-11-03,1.571429,3,3,ILI incidence,AT
6,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-11-10,2.571429,4,4,ILI incidence,AT
7,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-11-10,2.571429,4,4,ILI incidence,AT
8,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-10-20,-0.428571,1,1,ILI incidence,BE
9,respicast-quantileBaseline,2024-10-23-respicast-quantileBaseline.csv,2024-10-23,2024-10-23,2024-10-20,-0.428571,1,1,ILI incidence,BE



TFG horizon/date convention audit


,h,date_diff_weeks,size
0,1,1.0,12662
1,2,2.0,12536
2,3,3.0,12410
3,4,4.0,12284



Done. Send me these displayed tables, especially:
1) Duplicate prediction-key audit
2) Coverage audit
3) RespiCast horizon/date convention audit
